In [84]:
import sdv, anonymeter, pandas, streamlit, sklearn
print("Everything is ready!")

Everything is ready!


In [85]:
import pandas as pd
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

# Create dataset locally - no internet needed
data = {
    'Age': [22,38,26,35,35,28,54,2,27,14,4,58,20,39,14,
            55,2,31,35,34,15,28,8,38,19,40,66,28,42,45],
    'Fare': [7.25,71.28,7.92,53.1,8.05,8.46,51.86,21.07,
             11.13,30.07,16.7,26.55,8.05,31.27,7.85,16.0,
             29.12,13.0,18.0,23.0,12.0,9.5,11.5,7.75,
             8.05,13.0,10.0,7.75,26.0,14.5],
    'Pclass': [3,1,3,1,3,3,1,3,3,2,3,1,3,3,3,
               2,3,2,3,3,3,3,3,1,3,2,3,3,1,2],
    'Survived': [0,1,1,1,0,0,0,0,1,1,1,1,0,0,0,
                 1,0,1,0,1,0,1,1,1,0,1,0,1,1,0],
    'Sex': ['male','female','female','female','male','male',
            'male','male','female','female','female','female',
            'male','male','male','female','male','female',
            'female','female','male','female','male','female',
            'female','male','male','female','female','male']
}

df = pd.DataFrame(data)

print("=== ORIGINAL DATA ===")
print("Shape:", df.shape)
print(df.head(10))
print("\nData types:\n", df.dtypes)
print("\nBasic stats:\n", df.describe())

=== ORIGINAL DATA ===
Shape: (30, 5)
   Age   Fare  Pclass  Survived     Sex
0   22   7.25       3         0    male
1   38  71.28       1         1  female
2   26   7.92       3         1  female
3   35  53.10       1         1  female
4   35   8.05       3         0    male
5   28   8.46       3         0    male
6   54  51.86       1         0    male
7    2  21.07       3         0    male
8   27  11.13       3         1  female
9   14  30.07       2         1  female

Data types:
 Age           int64
Fare        float64
Pclass        int64
Survived      int64
Sex          object
dtype: object

Basic stats:
              Age       Fare     Pclass   Survived
count  30.000000  30.000000  30.000000  30.000000
mean   30.066667  19.326000   2.433333   0.533333
std    16.439929  15.569036   0.817200   0.507416
min     2.000000   7.250000   1.000000   0.000000
25%    19.250000   8.152500   2.000000   0.000000
50%    29.500000  13.000000   3.000000   1.000000
75%    38.750000  25.250000   

# UNDERSTANDING MY DATA - answer before training

# Q1: How many rows and columns do I have?
  Answer: 30 rows and 5 columns

# Q2: What are the data types?
# - Age = int64 (numerical)
# - Fare = float64 (numerical)  
# - Pclass = int64 (numerical/categorical?)
# - Survived = int64 (numerical/categorical?)
# - Sex = object (categorical - text)
# QUESTION: Why does it matter that Sex is text 
# and Age is a number? How will CTGAN treat them 
# differently?
Categorical Features (like Sex)
What it is: Categorical features are text labels such as Male or Female.
How CTGAN handles it: The model converts them into a numeric format using one-hot encoding or embedding, which allows it to understand each category separately.
Why it matters: This ensures that when CTGAN generates new rows, each value corresponds to a valid category instead of producing meaningless “in-between” values like 0.5 for sex.
Extra benefit: You can tell the model to generate data for a specific category, e.g., only Male entries, using conditional sampling.
Numerical Features (like Age)
What it is: Numerical features are continuous numbers like 28 or 42.
How CTGAN handles it: CTGAN normalizes these values and learns their real-number distribution so new values are realistic and span the same range as the original dataset.
Why it matters: The generated ages will be sensible numbers (like 30, 31.5, etc.) instead of being forced into discrete categories.
Key Takeaways
Treating categorical features as numbers can confuse the model and create unrealistic synthetic data.
Treating numerical features as categories limits the model’s ability to generate smooth, realistic values.
CTGAN uses different modeling and loss strategies internally for text vs number columns to preserve their natural patterns and relationships.
By keeping categorical and numerical data separate, CTGAN produces synthetic datasets that look realistic, respect category boundaries, and maintain the right value ranges for continuous features.

# Q3: Why did we dropna / use clean data?
 Answer: In short: dropping NaN or using clean data is necessary for CTGAN because it requires complete feature values to learn realistic distributions for both categorical and numerical columns. Without it, you risk corrupted synthetic data or unstable training.

# Q4: What is CTGAN actually trying to learn 
# from this data?
Answer: In essence, CTGAN is trying to learn:
The marginal distributions of each column (continuous or categorical).
The joint distributions and statistical dependencies between columns.
A mapping from latent noise to realistic tabular records, conditioned on rare or important features.
This allows users to generate synthetic datasets that maintain the key statistical properties of the original data, useful for privacy-preserving data sharing, model training augmentation, or testing.

# Q5: After training, the synthetic data will 
# contain NO real people's records - but it 
# will look statistically similar. 
# Why does that make it useful for AI training?
# Answer:
CTGAN produces synthetic datasets that appear statistically similar to the original, yet contain no identifiable personal information, allowing safe and effective use in research, analysis, and model development.However, statistical similarity alone doesn't prove privacy. A synthetic dataset can still leak information about real individuals through privacy attacks — which is exactly why SynthProof runs singling out, linkability and inference tests after generation. The compliance pack proves the synthetic data is safe, not just similar."

In [86]:
# Step 3 - See what CTGAN detects about our data

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

print("What CTGAN detected about our columns:")
print(metadata.to_dict())

What CTGAN detected about our columns:
{'columns': {'Age': {'sdtype': 'numerical'}, 'Fare': {'sdtype': 'numerical'}, 'Pclass': {'sdtype': 'categorical'}, 'Survived': {'sdtype': 'categorical'}, 'Sex': {'sdtype': 'categorical'}}, 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}


In [87]:
import json
print(json.dumps(metadata.to_dict(), indent=2))

{
  "columns": {
    "Age": {
      "sdtype": "numerical"
    },
    "Fare": {
      "sdtype": "numerical"
    },
    "Pclass": {
      "sdtype": "categorical"
    },
    "Survived": {
      "sdtype": "categorical"
    },
    "Sex": {
      "sdtype": "categorical"
    }
  },
  "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1"
}


In [88]:
# Train CTGAN
synthesizer = CTGANSynthesizer(
    metadata,
    epochs=50,
    verbose=True
)

print("Training started... watch the loss values drop")
synthesizer.fit(df)
print("\n✅ Training complete!")

Training started... watch the loss values drop


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+01.26) | Discrim. (+00.13):  74%|███████▍  | 37/50 [00:01<00:00, 23.24it/s]


KeyboardInterrupt: 

In [ ]:
# Generate synthetic data
synthetic_df = synthesizer.sample(num_rows=100)

print("=== YOUR FIRST SYNTHETIC DATASET ===")
print("Shape:", synthetic_df.shape)
print("\nFirst 10 synthetic people:")
print(synthetic_df.head(10))
print("\nSynthetic stats:")
print(synthetic_df.describe())

=== YOUR FIRST SYNTHETIC DATASET ===
Shape: (100, 5)

First 10 synthetic people:
   Age   Fare  Pclass  Survived     Sex
0    2  41.38       2         1  female
1   55  28.70       1         1  female
2   44  33.74       3         1    male
3   66  26.98       1         1  female
4   24  21.56       3         1    male
5   23  22.19       3         1  female
6   54  28.99       1         1    male
7    9  32.97       3         0    male
8   22  27.16       2         0    male
9   40  32.67       3         1  female

Synthetic stats:
              Age        Fare      Pclass    Survived
count  100.000000  100.000000  100.000000  100.000000
mean    39.670000   26.183900    2.320000    0.640000
std     20.288851    9.359274    0.839432    0.482418
min      2.000000    7.250000    1.000000    0.000000
25%     23.000000   21.390000    2.000000    0.000000
50%     41.500000   25.835000    3.000000    1.000000
75%     58.000000   30.305000    3.000000    1.000000
max     66.000000   71.280000

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Real vs Synthetic Data — SynthProof', 
             fontsize=14, fontweight='bold')

# Age comparison
axes[0,0].hist(df['Age'], bins=15, alpha=0.7, 
               color='#1B3A6B', label='Real')
axes[0,0].hist(synthetic_df['Age'], bins=15, 
               alpha=0.7, color='#0D7377', 
               label='Synthetic')
axes[0,0].set_title('Age Distribution')
axes[0,0].legend()

# Fare comparison
axes[0,1].hist(df['Fare'], bins=15, alpha=0.7, 
               color='#1B3A6B', label='Real')
axes[0,1].hist(synthetic_df['Fare'], bins=15, 
               alpha=0.7, color='#0D7377', 
               label='Synthetic')
axes[0,1].set_title('Fare Distribution')
axes[0,1].legend()

# Survived comparison
real_survived = df['Survived'].value_counts()
syn_survived = synthetic_df['Survived'].value_counts()
x = ['Not Survived (0)', 'Survived (1)']
axes[1,0].bar(x, [real_survived.get(0,0), 
               real_survived.get(1,0)], 
              alpha=0.7, color='#1B3A6B', 
              label='Real', width=0.4, 
              align='center')
axes[1,0].bar(x, [syn_survived.get(0,0), 
               syn_survived.get(1,0)], 
              alpha=0.7, color='#0D7377', 
              label='Synthetic', width=0.2, 
              align='edge')
axes[1,0].set_title('Survival Distribution')
axes[1,0].legend()

# Sex comparison
real_sex = df['Sex'].value_counts()
syn_sex = synthetic_df['Sex'].value_counts()
axes[1,1].bar(['Male', 'Female'], 
              [real_sex.get('male',0), 
               real_sex.get('female',0)],
              alpha=0.7, color='#1B3A6B', 
              label='Real', width=0.4)
axes[1,1].bar(['Male', 'Female'], 
              [syn_sex.get('male',0), 
               syn_sex.get('female',0)],
              alpha=0.7, color='#0D7377', 
              label='Synthetic', width=0.2,
              align='edge')
axes[1,1].set_title('Sex Distribution')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('/Users/shamna/Desktop/synthproof_comparison.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved as synthproof_comparison.png")

✅ Chart saved as synthproof_comparison.png


/var/folders/bd/mvt9cw1n1_386_2v3zhsbnmh0000gq/T/ipykernel_71550/2865026112.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
from anonymeter.evaluators import SinglingOutEvaluator

train = df.iloc[:20]
holdout = df.iloc[20:]

evaluator = SinglingOutEvaluator(
    ori=train,
    syn=synthetic_df[:20],
    control=holdout,
    n_attacks=10
)

evaluator.evaluate()
risk = evaluator.risk()
print("✅ Singling Out Risk Score:", risk)

✅ Singling Out Risk Score: PrivacyRisk(value=0.04095522899193511, ci=(0.0, 0.3492550908718834))


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/anonymeter/stats/confidence.py:230: UserWarning: Attack is as good or worse as baseline model. Estimated rates: attack = 0.21101311994515567, baseline = 0.21101311994515567. Analysis results cannot be trusted.
  self._sanity_check()


In [ ]:
from anonymeter.evaluators import LinkabilityEvaluator

evaluator2 = LinkabilityEvaluator(
    ori=train,
    syn=synthetic_df[:20],
    control=holdout,
    n_attacks=10,
    aux_cols=[['Age', 'Sex'], ['Pclass', 'Fare']]
)

evaluator2.evaluate()
risk2 = evaluator2.risk()
print("✅ Linkability Risk Score:", risk2)

✅ Linkability Risk Score: PrivacyRisk(value=0.08388748419471806, ci=(0.0, 0.3523631034973438))


In [ ]:
from anonymeter.evaluators import InferenceEvaluator

evaluator3 = InferenceEvaluator(
    ori=train,
    syn=synthetic_df[:20],
    control=holdout,
    n_attacks=10,
    secret='Survived',
    aux_cols=['Age', 'Sex', 'Pclass', 'Fare']
)

evaluator3.evaluate()
risk3 = evaluator3.risk()
print("✅ Inference Risk Score:", risk3)

✅ Inference Risk Score: PrivacyRisk(value=0.16889810877902983, ci=(0.0, 0.9368311674458794))


In [ ]:
# ============================================
# SYNTHPROOF - COMPLETE PIPELINE SUMMARY
# May 5 2026
# ============================================

print("=" * 50)
print("SYNTHPROOF PRIVACY EVALUATION REPORT")
print("=" * 50)
print(f"\nDataset: Titanic (demo)")
print(f"Original rows: {len(train)}")
print(f"Synthetic rows: {len(synthetic_df[:20])}")
print("\n--- ICO PRIVACY ATTACK RESULTS ---")
print(f"1. Singling Out Risk:  {risk.value:.3f}  🟡 AMBER")
print(f"2. Linkability Risk:   {risk2.value:.3f}  🟢 GREEN")
print(f"3. Inference Risk:     {risk3.value:.3f}  🟢 GREEN")
print("\n--- OVERALL ASSESSMENT ---")
print("Overall Privacy Risk: LOW 🟢")
print("ICO Anonymisation Standard: LIKELY MET")
print("\nNote: Results based on small demo dataset.")
print("Production use requires 500+ rows minimum.")
print("=" * 50)
print("Generated by SynthProof")
print("github.com/Shamnakottakkodan/synthproof")

SYNTHPROOF PRIVACY EVALUATION REPORT

Dataset: Titanic (demo)
Original rows: 20
Synthetic rows: 20

--- ICO PRIVACY ATTACK RESULTS ---
1. Singling Out Risk:  0.041  🟡 AMBER
2. Linkability Risk:   0.084  🟢 GREEN
3. Inference Risk:     0.169  🟢 GREEN

--- OVERALL ASSESSMENT ---
Overall Privacy Risk: LOW 🟢
ICO Anonymisation Standard: LIKELY MET

Note: Results based on small demo dataset.
Production use requires 500+ rows minimum.
Generated by SynthProof
github.com/Shamnakottakkodan/synthproof
